# FitBuddy Food Model Training (No Google Drive Method)

**Objective:** Fine-tune a pre-trained image recognition model (MobileNetV2) to recognize the specific food items from your dataset.

**Instructions:**
1. On the left side of the Colab window, click the **folder icon** to open the Files tab.
2. Click the **'Upload to session storage'** icon (the paper icon with an up arrow) and select your `images.zip` file.
3. Wait for the upload to complete. You will see `images.zip` appear in the file list.
4. Once the upload is finished, run each cell in this notebook in order by clicking the "Play" button.

## Step 1: Environment Setup

This cell imports the necessary libraries. We'll use TensorFlow for the model training.

In [ ]:
import tensorflow as tf
import os

print(f'TensorFlow Version: {tf.__version__}')

## Step 2: Unzip Your Dataset

This cell will find the `images.zip` file you uploaded to this session and extract it.

In [ ]:
!unzip /content/images.zip -d /content/dataset

## Step 3: Load and Prepare Data

Now we'll load the images from the folder you unzipped. We'll define some parameters and use a Keras utility to automatically load, resize, and label the images based on their folder names.

In [ ]:
IMG_SIZE = (224, 224) # The input size for MobileNetV2
BATCH_SIZE = 32
DATA_DIR = '/content/dataset/images'

# Load the data from directories and create training and validation splits
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f'Found the following {len(class_names)} classes:', class_names)

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

## Step 4: Build the Fine-Tuning Model (Corrected)

Here, we perform the magic of **transfer learning**. This version uses the Keras Functional API to ensure the new final layer is trainable.

In [ ]:
# Create the base model from the pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# Freeze the base model
base_model.trainable = False

# Define the input layer
inputs = tf.keras.Input(shape=(224, 224, 3))

# Pre-process the inputs
x = tf.keras.layers.Rescaling(1./255)(inputs)

# Run the frozen base model in inference mode
x = base_model(x, training=False)

# Add our new classification head
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(len(class_names))(x)

# Create the final model
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.summary()

## Step 5: Train the Model!

This is where the fine-tuning happens. The model will learn to associate the features extracted by MobileNetV2 with your specific food classes. This should only take a few minutes on a GPU.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

## Step 6: Convert and Save the Model for Web Usage

The model is currently in a Keras/TensorFlow format. We need to convert it to the TensorFlow.js format (`model.json` + binary weight files) that our Node.js backend can use.

In [ ]:
!pip install -q tensorflowjs # Install the converter

import tensorflowjs as tfjs
import json

# Create a directory to save the model
os.makedirs('tfjs_model', exist_ok=True)

# Save the model
tfjs.converters.save_keras_model(model, 'tfjs_model')

# Also save the class names (labels) to a file, as the model itself doesn't store them
with open('tfjs_model/class_names.json', 'w') as f:
    json.dump(class_names, f)

print(f'Model saved to \tfjs_model\ directory. Labels saved to \class_names.json\.')

## Step 7: Download Your Trained Model

This final cell will zip the converted model files and trigger a download. 

**Next Steps:**
1. Unzip the downloaded `tfjs_model.zip` file.
2. You will see a folder named `tfjs_model` containing `model.json`, `class_names.json`, and one or more `groupX-shard...` files.
3. Move this entire `tfjs_model` folder into the `Backend` directory of your FitBuddy project.
4. Let me know once you have done this, and I will modify the backend code to use your new model!

In [ ]:
!zip -r /content/tfjs_model.zip /content/tfjs_model

from google.colab import files
files.download('/content/tfjs_model.zip')